In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))


In [4]:
from src.db import get_connection
import pandas as pd

conn = get_connection()
pd.read_sql_query("SELECT * FROM metrics ORDER BY date", conn)


DatabaseError: Execution failed on sql 'SELECT * FROM metrics ORDER BY date': no such table: metrics

In [ ]:
from src.db import get_connection, init_db, insert_sample_rows
import pandas as pd


conn = get_connection("../data/bootcamp_stage5.db")
init_db(conn)
insert_sample_rows(conn)

pd.read_sql_query("SELECT * FROM metrics ORDER BY date", conn)


,id,date,revenue,risk_score
0,1,2025-08-14,1200.0,0.35
1,2,2025-08-15,1525.0,0.30
2,3,2025-08-16,980.0,0.55
3,4,2025-08-17,1730.0,0.28


In [6]:
# src/db.py
from __future__ import annotations
import sqlite3
from pathlib import Path

# anchor to the homework5 root (two levels up from this file: src/ → homework5/)
ROOT = Path(__file__).resolve().parents[1]
DB_PATH = ROOT / "data" / "bootcamp_stage5.db"

def get_connection(db_path: str | Path = DB_PATH) -> sqlite3.Connection:
    db_path = Path(db_path)
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


NameError: name '__file__' is not defined

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent)) 

from src.db import get_connection, init_db, insert_sample_rows
import pandas as pd

conn = get_connection()
init_db(conn)
insert_sample_rows(conn)

pd.read_sql_query("SELECT * FROM metrics ORDER BY date", conn)

,id,date,revenue,risk_score
0,1,2025-08-14,1200.0,0.35
1,2,2025-08-15,1525.0,0.30
2,3,2025-08-16,980.0,0.55
3,4,2025-08-17,1730.0,0.28


In [9]:
conn = get_connection()

# list all tables inside your SQLite DB
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)


,name
0,metrics
1,sqlite_sequence


In [10]:
pd.read_sql_query("PRAGMA table_info(metrics);", conn)


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,date,TEXT,1,None,0
2,2,revenue,REAL,1,None,0
3,3,risk_score,REAL,1,None,0


In [11]:
# env + paths
import os, pathlib, textwrap, datetime as dt
from dotenv import load_dotenv

import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

load_dotenv(dotenv_path=pathlib.Path("../.env"))  
RAW_DIR = pathlib.Path(os.getenv("DATA_DIR_RAW", "data/raw")).resolve()
PROC_DIR = pathlib.Path(os.getenv("DATA_DIR_PROCESSED", "data/processed")).resolve()

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROC_DIR


(WindowsPath('C:/Users/User/bootcamp_Khushi_Khanna/homework/homework5/notebooks/data/raw'),
 WindowsPath('C:/Users/User/bootcamp_Khushi_Khanna/homework/homework5/notebooks/data/processed'))

In [12]:
import pandas as pd

def write_df(df: pd.DataFrame, path: pathlib.Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df.to_csv(path, index=False)
    elif suffix == ".parquet":
        # parquet engine check
        try:
            import pyarrow  # noqa: F401
        except ModuleNotFoundError:
            raise RuntimeError(
                "Parquet support requires 'pyarrow'. Install with: pip install pyarrow"
            )
        df.to_parquet(path, index=False)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

def read_df(path: pathlib.Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    elif suffix == ".parquet":
        try:
            import pyarrow  # noqa: F401
        except ModuleNotFoundError:
            raise RuntimeError(
                "Parquet support requires 'pyarrow'. Install with: pip install pyarrow"
            )
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


In [13]:
from src.db import get_connection

conn = get_connection()
df = pd.read_sql_query("SELECT date, revenue, risk_score FROM metrics ORDER BY date", conn)
conn.close()
df.head()


,date,revenue,risk_score
0,2025-08-14,1200.0,0.35
1,2025-08-15,1525.0,0.30
2,2025-08-16,980.0,0.55
3,2025-08-17,1730.0,0.28


In [14]:
stamp = dt.datetime.now().strftime("%Y%m%d-%H%M")

csv_path = RAW_DIR / f"metrics_{stamp}.csv"
parq_path = PROC_DIR / f"metrics_{stamp}.parquet"

write_df(df, csv_path)
write_df(df, parq_path)

csv_path, parq_path


(WindowsPath('C:/Users/User/bootcamp_Khushi_Khanna/homework/homework5/notebooks/data/raw/metrics_20250817-2235.csv'),
 WindowsPath('C:/Users/User/bootcamp_Khushi_Khanna/homework/homework5/notebooks/data/processed/metrics_20250817-2235.parquet'))

In [ ]:
df_csv = read_df(csv_path)
df_parq = read_df(parq_path)

print("Shapes:", df.shape, df_csv.shape, df_parq.shape)
print()
print("Dtypes original:\n", df.dtypes)
print("Dtypes csv:\n", df_csv.dtypes)
print("Dtypes parquet:\n", df_parq.dtypes)


def validate_shapes(*frames: pd.DataFrame) -> bool:
    base = frames[0].shape
    return all(f.shape == base for f in frames)

def validate_required_columns(df: pd.DataFrame, cols: list[str]) -> bool:
    return all(c in df.columns for c in cols)

print("\nValid shapes:", validate_shapes(df, df_csv, df_parq))
print("Has required cols:", validate_required_columns(df_parq, ["date","revenue","risk_score"]))


Shapes: (4, 3) (4, 3) (4, 3)

Dtypes original:
 date           object
revenue       float64
risk_score    float64
dtype: object
Dtypes csv:
 date           object
revenue       float64
risk_score    float64
dtype: object
Dtypes parquet:
 date           object
revenue       float64
risk_score    float64
dtype: object

Valid shapes: True
Has required cols: True
